In [119]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

In [120]:
text_df = pd.read_csv('./Data/depression_dataset_reddit_cleaned.csv')
tab_df = pd.read_csv('./Data/Student_Mental_health.csv')

print(text_df.head())
print(tab_df.head())
print(tab_df.columns)

                                          clean_text  is_depression
0  we understand that most people who reply immed...              1
1  welcome to r depression s check in post a plac...              1
2  anyone else instead of sleeping more when depr...              1
3  i ve kind of stuffed around a lot in my life d...              1
4  sleep is my greatest and most comforting escap...              1
        Timestamp Choose your gender   Age What is your course?  \
0  8/7/2020 12:02             Female  18.0          Engineering   
1  8/7/2020 12:04               Male  21.0    Islamic education   
2  8/7/2020 12:05               Male  19.0                  BIT   
3  8/7/2020 12:06             Female  22.0                 Laws   
4  8/7/2020 12:13               Male  23.0         Mathemathics   

  Your current year of Study What is your CGPA? Marital status  \
0                     year 1        3.00 - 3.49             No   
1                     year 2        3.00 - 3.49          

In [121]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#\w+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [122]:
text_df = text_df[['clean_text', 'is_depression']].copy()
text_df['clean_text'] = text_df['clean_text'].apply(clean_text)

print(text_df.head())

                                          clean_text  is_depression
0  we understand that most people who reply immed...              1
1  welcome to r depression s check in post a plac...              1
2  anyone else instead of sleeping more when depr...              1
3  i ve kind of stuffed around a lot in my life d...              1
4  sleep is my greatest and most comforting escap...              1


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2
    max_df=0.95,
    sublinear_tf=True
)

X_text = tfidf.fit_transform(text_df['clean_text'])
y_text = text_df['is_depression']

In [124]:
tab_df = pd.read_csv('./Data/Student_Mental_health.csv')  # local path
tab_df = tab_df.copy()

# convert empty strings to NaN
tab_df = tab_df.replace(r'^\s*$', np.nan, regex=True)

# drop rows with missing values
tab_df = tab_df.dropna().reset_index(drop=True)

# drop timestamp if present
if 'Timestamp' in tab_df.columns:
    tab_df = tab_df.drop('Timestamp', axis=1)

print(tab_df.isna().sum())
print(tab_df.shape)

Choose your gender                              0
Age                                             0
What is your course?                            0
Your current year of Study                      0
What is your CGPA?                              0
Marital status                                  0
Do you have Depression?                         0
Do you have Anxiety?                            0
Do you have Panic attack?                       0
Did you seek any specialist for a treatment?    0
dtype: int64
(100, 10)


In [125]:
def clean_categorical(col):
    return col.astype(str).str.lower().str.strip()
for col in tab_df.columns:
    if tab_df[col].dtype == 'object':
        tab_df[col] = clean_categorical(tab_df[col])

In [126]:
label_encoders = {}

for col in tab_df.columns:
    if tab_df[col].dtype == 'object' or tab_df[col].dtype == 'string':
        le = LabelEncoder()
        tab_df[col] = le.fit_transform(tab_df[col].astype(str))
        label_encoders[col] = le

print("\nAfter encoding:")
print(tab_df.dtypes)


After encoding:
Choose your gender                                int64
Age                                             float64
What is your course?                              int64
Your current year of Study                        int64
What is your CGPA?                                int64
Marital status                                    int64
Do you have Depression?                           int64
Do you have Anxiety?                              int64
Do you have Panic attack?                         int64
Did you seek any specialist for a treatment?      int64
dtype: object


In [127]:
target_col = 'Do you have Depression?'

X_tab = tab_df.drop(target_col, axis=1)
y_tab = tab_df[target_col]

feature_columns = X_tab.columns.tolist()

print(X_tab.head())
print(X_tab.dtypes)

   Choose your gender   Age  What is your course?  Your current year of Study  \
0                   0  18.0                    17                           3   
1                   1  21.0                    25                           4   
2                   1  19.0                     4                           0   
3                   0  22.0                    33                           5   
4                   1  23.0                    37                           6   

   What is your CGPA?  Marital status  Do you have Anxiety?  \
0                   3               0                     0   
1                   3               0                     1   
2                   3               0                     1   
3                   3               1                     0   
4                   3               0                     0   

   Do you have Panic attack?  Did you seek any specialist for a treatment?  
0                          1                             

In [128]:
print(X_tab.select_dtypes(include=['object', 'string']).columns)

Index([], dtype='str')


In [129]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_tab_scaled = scaler.fit_transform(X_tab)

print(X_tab_scaled[:5])

[[-0.57735027 -1.01861394 -0.01362317  0.0916243  -0.35830874 -0.43643578
  -0.71774056  1.42488702 -0.25264558]
 [ 1.73205081  0.18922868  0.55998412  0.79642662 -0.35830874 -0.43643578
   1.39326109 -0.70181003 -0.25264558]
 [ 1.73205081 -0.61599974 -0.94573503 -2.02278266 -0.35830874 -0.43643578
   1.39326109  1.42488702 -0.25264558]
 [-0.57735027  0.59184288  1.13359142  1.50122894 -0.35830874  2.29128785
  -0.71774056 -0.70181003 -0.25264558]
 [ 1.73205081  0.99445709  1.42039507  2.20603126 -0.35830874 -0.43643578
  -0.71774056 -0.70181003 -0.25264558]]


In [130]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_text, y_text, test_size=0.2, random_state=42, stratify=y_text
)

In [131]:
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_tab_scaled, y_tab, test_size=0.2, random_state=42, stratify=y_tab
)

In [132]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Text Accuracy:", accuracy_score(y_test, pred))

Text Accuracy: 0.9566903684550744


In [133]:
print("NaN in tab_df:", tab_df.isna().sum())
print("\nNaN in X_tab:", X_tab.isna().sum())
print("\nAny NaN in X_tab?", X_tab.isna().any().any())

NaN in tab_df: Choose your gender                              0
Age                                             0
What is your course?                            0
Your current year of Study                      0
What is your CGPA?                              0
Marital status                                  0
Do you have Depression?                         0
Do you have Anxiety?                            0
Do you have Panic attack?                       0
Did you seek any specialist for a treatment?    0
dtype: int64

NaN in X_tab: Choose your gender                              0
Age                                             0
What is your course?                            0
Your current year of Study                      0
What is your CGPA?                              0
Marital status                                  0
Do you have Anxiety?                            0
Do you have Panic attack?                       0
Did you seek any specialist for a treatment?    0
dtype: 

In [134]:
model2 = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)
model2.fit(X_train_t, y_train_t)

pred2 = model2.predict(X_test_t)

print("Tabular Accuracy:", accuracy_score(y_test_t, pred2))

Tabular Accuracy: 0.8


In [135]:
print("Text model and tabular model are trained successfully.")
print("Hybrid fusion will be used only for live user prediction.")

Text model and tabular model are trained successfully.
Hybrid fusion will be used only for live user prediction.


In [136]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

input_dim = X_train_t.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(16, activation='relu')(input_layer)
encoded = Dense(8, activation='relu')(encoded)

decoded = Dense(16, activation='relu')(encoded)
decoded = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(
    X_train_t, X_train_t,
    epochs=50,
    batch_size=16,
    validation_data=(X_test_t, X_test_t),
    verbose=0
)

In [137]:
recon_train = autoencoder.predict(X_train_t, verbose=0)
train_mse = np.mean(np.power(X_train_t - recon_train, 2), axis=1)

threshold = np.percentile(train_mse, 95)

recon_all = autoencoder.predict(X_tab_scaled, verbose=0)
mse = np.mean(np.power(X_tab_scaled - recon_all, 2), axis=1)

anomalies = mse > threshold

print("Threshold:", threshold)
print("Total anomalies:", np.sum(anomalies))

Threshold: 1.2333703508693374
Total anomalies: 5


In [138]:
from sklearn.metrics import classification_report, confusion_matrix

print("Text Model Report:")
print(classification_report(y_test, pred))
print("Text Confusion Matrix:")
print(confusion_matrix(y_test, pred))

print("Tabular Model Report:")
print(classification_report(y_test_t, pred2))
print("Tabular Confusion Matrix:")
print(confusion_matrix(y_test_t, pred2))

Text Model Report:
              precision    recall  f1-score   support

           0       0.94      0.98      0.96       780
           1       0.98      0.93      0.96       767

    accuracy                           0.96      1547
   macro avg       0.96      0.96      0.96      1547
weighted avg       0.96      0.96      0.96      1547

Text Confusion Matrix:
[[763  17]
 [ 50 717]]
Tabular Model Report:
              precision    recall  f1-score   support

           0       0.85      0.85      0.85        13
           1       0.71      0.71      0.71         7

    accuracy                           0.80        20
   macro avg       0.78      0.78      0.78        20
weighted avg       0.80      0.80      0.80        20

Tabular Confusion Matrix:
[[11  2]
 [ 2  5]]


In [139]:
def encode_user_input(user_df, label_encoders):
    user_df = user_df.copy()
    for col, encoder in label_encoders.items():
        if col in user_df.columns:
            user_df[col] = encoder.transform(user_df[col].astype(str))
    return user_df

In [140]:
def predict_user_mental_health_app(
    user_text,
    age,
    gender,
    course,
    year_of_study,
    cgpa,
    marital_status,
    anxiety,
    panic_attack,
    treatment
):
    # -------- TEXT --------
    cleaned_text = clean_text(user_text)
    text_vector = tfidf.transform([cleaned_text])
    text_prob = model.predict_proba(text_vector)[0][1]

    # -------- TABULAR --------
    user_tab_df = pd.DataFrame([{
        'Choose your gender': gender,
        'Age': age,
        'What is your course?': course,
        'Your current year of Study': year_of_study,
        'What is your CGPA?': cgpa,
        'Marital status': marital_status,
        'Do you have Anxiety?': anxiety,
        'Do you have Panic attack?': panic_attack,
        'Did you seek any specialist for a treatment?': treatment
    }])

    user_tab_df = encode_user_input(user_tab_df, label_encoders)
    user_tab_df = user_tab_df[feature_columns]

    user_tab_scaled = scaler.transform(user_tab_df)
    tab_prob = model2.predict_proba(user_tab_scaled)[0][1]

    # -------- FUSION --------
    final_prob = (text_prob + tab_prob) / 2
    risk_score = final_prob * 100

    if risk_score < 30:
        risk_level = "Low"
    elif risk_score < 70:
        risk_level = "Medium"
    else:
        risk_level = "High"

    # -------- ANOMALY --------
    reconstructed = autoencoder.predict(user_tab_scaled, verbose=0)
    user_mse = np.mean((user_tab_scaled - reconstructed) ** 2, axis=1)[0]
    anomaly = user_mse > threshold

    # -------- INTERPRETATION --------
    if risk_score < 30:
        message = "Low mental health risk."
    elif risk_score < 70:
        message = "Moderate mental health risk."
    else:
        message = "High mental health risk. Consider professional support."

    if anomaly:
        message += " Unusual behavior pattern detected."

    return {
    "text_probability": round(float(text_prob), 4),
    "tabular_probability": round(float(tab_prob), 4),
    "risk_score": round(float(risk_score), 2),
    "risk_level": risk_level,
    "anomaly": bool(anomaly),
    "message": message
}

In [141]:
print(tab_df_original['Choose your gender'].unique())
print(tab_df_original['What is your course?'].unique())
print(tab_df_original['Your current year of Study'].unique())
print(tab_df_original['What is your CGPA?'].unique())
print(tab_df_original['Marital status'].unique())
print(tab_df_original['Do you have Anxiety?'].unique())

<StringArray>
['Female', 'Male']
Length: 2, dtype: str
<StringArray>
[            'Engineering',       'Islamic education',
                     'BIT',                    'Laws',
            'Mathemathics',        'Pendidikan islam',
                     'BCS',         'Human Resources',
                   'Irkhs',              'Psychology',
                   'KENMS',             'Accounting ',
                     'ENM',          'Marine science',
                     'KOE',         'Banking Studies',
 'Business Administration',                     'Law',
                  'KIRKHS',              'Usuluddin ',
                   'TAASL',                  'Engine',
                     'ALA',      'Biomedical science',
                     'koe',                  'Kirkhs',
                    'BENL',                    'Benl',
                      'IT',                     'CTS',
                   'engin',                   'Econs',
                    'MHSC',                  'Malco

In [142]:
result = predict_user_mental_health_app(
    user_text="I feel very tired and hopeless these days.",
    age=21,
    gender="Female",
    course="Engineering",   # must match dataset
    year_of_study="Year 2",
    cgpa="3.00 - 3.49",
    marital_status="No",
    anxiety="Yes",
    panic_attack="No",
    treatment="No"
)

print(result)

{'text_probability': 0.3745, 'tabular_probability': 0.5883, 'risk_score': 48.14, 'risk_level': 'Medium', 'anomaly': False, 'message': 'Moderate mental health risk.'}


In [143]:
import pickle

# models
with open("text_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("tabular_model.pkl", "wb") as f:
    pickle.dump(model2, f)

# preprocessing
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("label_encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

with open("feature_columns.pkl", "wb") as f:
    pickle.dump(feature_columns, f)

with open("threshold.pkl", "wb") as f:
    pickle.dump(threshold, f)

In [144]:
autoencoder.save("autoencoder_model.h5")